《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구할 수 있습니다. 두 링크는 원저자의 제휴 링크입니다. 책이 도움이 되었다면 Green Tea Press의 [무료 공개 교육 자료 제작을 후원](https://greenteapress.com/support/?book=ts3)하는 것도 생각해 주세요.

# 확률질량함수

앞 장에서는 값들과 그 도수, 즉 각 값이 나타나는 횟수를 담은 `FreqTab` 객체로 분포를 표현했습니다. 이 장에서는 분포를 설명하는 또 다른 방법인 확률질량함수(PMF)를 소개하겠습니다.

PMF를 표현할 때는 값들과 그 확률을 담은 `Pmf` 객체를 사용합니다. `Pmf`로 분포의 평균과 분산을 계산하고, 왼쪽이나 오른쪽으로 치우친 정도를 나타내는 왜도도 구해 보겠습니다. 마지막으로 “관측의 역설”이라는 현상 때문에 표본이 분포를 편향되게 보여줄 수 있다는 점을 살펴보겠습니다.

[이 노트북의 원문을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/v3/nb/chap03.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thinkstats import decorate

## PMF

`Pmf` 객체는 도수 대신 확률을 담는다는 점을 빼면 `FreqTab`과 비슷합니다. 따라서 `FreqTab`에서 출발해 `Pmf`를 만들 수 있습니다. 짧은 시퀀스에 들어 있는 값들의 분포를 나타내는 `FreqTab`을 보겠습니다.

In [4]:
from empiricaldist import FreqTab

ftab = FreqTab.from_seq([1, 2, 2, 3, 5])
ftab

도수의 합은 원래 시퀀스의 길이와 같습니다.

In [5]:
n = ftab.sum()
n

도수를 `n`으로 나누면 개수가 아니라 비율을 나타내게 됩니다.

In [6]:
pmf = ftab / n
pmf

이 결과는 시퀀스에서 1이 20%, 2가 40%를 차지한다는 뜻입니다.

이 비율을 확률로 생각할 수도 있습니다. 원래 시퀀스에서 값 하나를 무작위로 고를 때, 1을 고를 확률은 0.2이고 2를 고를 확률은 0.4입니다.

모든 도수를 `n`으로 나누었으므로 확률의 합은 1입니다. 이처럼 확률의 합을 1로 맞춘 분포를 **정규화한**(normalized) 분포라고 합니다.

In [7]:
pmf.sum()

정규화한 `FreqTab` 객체는 **확률질량함수**(probability mass function, PMF)를 나타냅니다. 이산적인 값에 대응하는 확률을 “확률질량”이라고도 부르기 때문에 붙은 이름입니다.

`empiricaldist` 라이브러리는 확률질량함수를 나타내는 `Pmf` 객체를 제공합니다. 따라서 `FreqTab`을 만든 뒤 정규화하는 대신, 바로 `Pmf`를 만들 수 있습니다.

In [8]:
from empiricaldist import Pmf

pmf = Pmf.from_seq([1, 2, 2, 3, 5])
pmf

`Pmf`는 전체 확률의 합이 1이 되도록 정규화되어 있습니다.

In [9]:
pmf.sum()

`Pmf`와 `FreqTab`은 여러 면에서 비슷합니다. 어떤 값의 확률을 조회할 때는 대괄호 연산자를 쓸 수 있습니다.

In [10]:
pmf[2]

괄호로 `Pmf`를 함수처럼 호출해도 됩니다.

In [11]:
pmf(2)

어떤 값에 확률을 지정할 때는 대괄호 연산자를 써야 합니다.

In [12]:
pmf[2] = 0.2
pmf(2)

기존 `Pmf`에서 어떤 값에 대응하는 확률을 늘릴 수 있습니다.

In [13]:
pmf[2] += 0.3
pmf[2]

확률에 어떤 수를 곱할 수도 있습니다.

In [14]:
pmf[2] *= 0.5
pmf[2]

`Pmf`를 수정하면 정규화가 풀릴 수 있습니다. 확률의 합이 더 이상 1이 아닐 수 있다는 뜻입니다.

In [15]:
pmf.sum()

`normalize` 메서드는 모든 확률을 그 합으로 나누어 다시 정규화합니다. 반환값은 정규화하기 전의 합입니다.

In [16]:
pmf.normalize()

`Pmf`의 `copy` 메서드로 복사본을 만들 수 있습니다. 복사본을 수정해도 원본에는 영향을 주지 않습니다.

In [17]:
pmf.copy()

`FreqTab`처럼 `Pmf`에도 값에 접근하는 `qs` 속성이 있습니다. 확률에는 `ps` 속성으로 접근합니다.

막대그래프를 그리는 `bar` 메서드와 선 그래프를 그리는 `plot` 메서드도 있습니다.

## PMF 요약하기

[1장의 요약 통계량](/posts/statistics/think-stats-01#요약-통계량)에서는 표본의 값들을 모두 더한 뒤 개수로 나누어 평균을 구했습니다. 간단한 예를 보겠습니다.

In [18]:
seq = [1, 2, 2, 3, 5]

n = len(seq)
mean = np.sum(seq) / n
mean

이 시퀀스에 들어 있는 값들의 PMF를 구해 보겠습니다.

In [19]:
pmf = Pmf.from_seq(seq)

`Pmf`만 있어도 평균을 구할 수 있지만, 계산 과정은 다릅니다. 각 값에 그 확률을 곱한 뒤 모두 더해야 합니다.

In [20]:
mean = np.sum(pmf.ps * pmf.qs)
mean

여기서는 `n`으로 나눌 필요가 없습니다. `Pmf`를 정규화할 때 이미 나누었기 때문입니다. `mean` 메서드도 같은 일을 합니다.

In [21]:
pmf.mean()

분산을 구하려면 먼저 `Pmf`의 각 값에서 평균을 빼서 편차를 계산합니다.

In [22]:
deviations = pmf.qs - mean

그런 다음 편차의 제곱에 확률을 곱하고 모두 더합니다.

In [23]:
var = np.sum(pmf.ps * deviations**2)
var

`var` 메서드도 같은 일을 합니다.

In [24]:
pmf.var()

분산에서 표준편차를 구하는 방법은 앞에서와 같습니다.

In [25]:
np.sqrt(var)

`std` 메서드를 사용해도 됩니다.

In [26]:
pmf.std()

`Pmf`에는 확률이 가장 큰 값을 찾는 `mode` 메서드도 있습니다.

In [27]:
pmf.mode()

다른 메서드들은 앞으로 살펴보겠습니다. 시작하는 데는 이 정도면 충분합니다.

## 수업 규모의 역설

`Pmf`로 무엇을 할 수 있는지 보여주는 예로, 제가 “수업 규모의 역설”이라고 부르는 현상을 생각해 보겠습니다.

미국의 많은 대학에서 학생 대 교수 비율은 약 10 대 1입니다. 그런데 학생들은 수강생이 10명보다 많은 수업이 많고, 때로는 훨씬 많다는 데 놀랍니다. 이 차이에는 두 가지 이유가 있습니다.

- 학생은 보통 한 학기에 4~5개 수업을 듣지만 교수는 흔히 1~2개 수업을 맡습니다.
- 작은 수업에는 학생이 적고, 큰 수업에는 학생이 많습니다.

첫 번째 이유는 한번 짚어 주면 분명합니다. 두 번째는 조금 더 생각해 봐야 합니다. 예를 들어 어느 대학이 한 학기에 65개 수업을 개설했고, 규모 구간별 수업 수가 다음과 같다고 하겠습니다.

In [28]:
ranges = pd.interval_range(start=5, end=50, freq=5, closed="left")
ranges.name = "class size"

data = pd.DataFrame(index=ranges)
data["count"] = [8, 8, 14, 4, 6, 12, 8, 3, 2]
data

Pandas의 `interval_range` 함수는 각 레이블이 값의 구간을 나타내는 `Index`를 만듭니다. `[5, 10)`이라는 표기는 `5`를 포함하고 `10`은 포함하지 않는다는 뜻입니다. 각 구간에 속한 실제 수업 규모는 모르므로, 구간의 가운데에 있는 값으로 대표한다고 가정하겠습니다.

> 번역자 주: 원문 코드는 정수 수업 규모를 대표하도록 구간의 왼쪽 끝에 2를 더해 7, 12, …, 47을 사용합니다. 아래 평균과 그림도 이 값을 기준으로 합니다.

In [29]:
sizes = ranges.left + 2
sizes

이제 수업 규모의 분포를 나타내는 `Pmf`를 만들겠습니다. 값과 도수를 알고 있으므로 개수, 규모, 이름을 인수로 넘겨 `Pmf`를 바로 만들 수 있습니다. 새 `Pmf`를 정규화하면 원래 개수의 합이 반환됩니다.

In [30]:
counts = data["count"]
actual_pmf = Pmf(counts, sizes, name="actual")
actual_pmf.normalize()

대학에 평균 수업 규모를 물으면 이 분포의 평균인 23.7명이라고 답할 것입니다.

In [31]:
actual_pmf.mean()

하지만 학생들을 조사해 듣는 수업에 학생이 몇 명인지 묻고 평균을 내면, 더 큰 값이 나옵니다. 얼마나 커지는지 보겠습니다.

다음 함수는 실제 수업 규모의 `Pmf`를 받아 학생이 경험하는 수업 규모의 `Pmf`를 만듭니다. 두 분포의 값은 같지만, 확률에는 그 값을 곱합니다. 규모가 `x`인 수업에는 그 수업을 관측하는 학생이 `x`명 있기 때문입니다. 따라서 어떤 수업을 관측할 확률은 그 수업의 규모에 비례합니다.

In [32]:
def bias(pmf, name):
    # multiply each probability by class size
    ps = pmf.ps * pmf.qs

    # make a new Pmf and normalize it
    new_pmf = Pmf(ps, pmf.qs, name=name)
    new_pmf.normalize()
    return new_pmf

이제 학생들이 관측하는 편향된 `Pmf`를 계산할 수 있습니다.

In [33]:
observed_pmf = bias(actual_pmf, name="observed")

두 분포를 그리면 다음과 같습니다.

In [34]:
from thinkstats import two_bar_plots

two_bar_plots(actual_pmf, observed_pmf, width=2)
decorate(xlabel="Class size", ylabel="PMF")

학생이 관측하는 분포에는 작은 수업이 적고 큰 수업이 많습니다. 편향된 평균은 29.1명으로, 실제 평균보다 거의 25% 큽니다.

In [35]:
observed_pmf.mean()

이 연산을 거꾸로 할 수도 있습니다. 어떤 대학의 수업 규모 분포를 알고 싶지만 믿을 만한 자료를 구할 수 없다고 하겠습니다. 학생을 무작위로 뽑아 수강 중인 수업의 학생 수를 묻는 방법이 있습니다.

결과에는 앞서 본 이유로 편향이 생기지만, 실제 분포를 추정하는 데 사용할 수 있습니다. 다음 함수는 확률을 수업 규모로 나누어 `Pmf`의 편향을 보정합니다.

In [36]:
def unbias(pmf, name):
    # divide each probability by class size
    ps = pmf.ps / pmf.qs

    new_pmf = Pmf(ps, pmf.qs, name=name)
    new_pmf.normalize()
    return new_pmf

결과는 다음과 같습니다.

In [37]:
debiased_pmf = unbias(observed_pmf, "debiased")
debiased_pmf.mean()

편향을 보정한 `Pmf`의 평균은 처음에 사용한 실제 분포의 평균과 같습니다.

이 예제가 흥미로웠다면 《Probably Overthinking It》의 2장도 마음에 들 수 있습니다. 이 예제와 함께 “관측의 역설”(inspection paradox)이라고 부르는 현상의 여러 사례가 나옵니다.

## NSFG 자료

앞 장에서는 첫아이와 다른 아이의 임신 기간을 도수분포표로 그렸습니다. 하지만 두 집단의 크기가 달라 직접 비교할 수 없었습니다. PMF는 정규화되어 있으므로 비교할 수 있습니다. NSFG 자료를 다시 불러와 임신 기간의 분포를 나타내는 `Pmf` 객체를 만들겠습니다.

다음 셀들은 데이터 파일을 내려받고, 자료를 읽는 데 필요한 `statadict`를 설치합니다.

In [38]:
try:
    import statadict
except ImportError:
    %pip install statadict

In [39]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/nsfg.py")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

`nsfg` 모듈의 `get_nsfg_groups` 함수는 자료를 읽고, 생존 출산에 해당하는 행을 고른 뒤, 첫아이와 나머지로 나눕니다. `DataFrame` 객체 세 개를 반환합니다.

In [40]:
from nsfg import get_nsfg_groups

live, firsts, others = get_nsfg_groups()

`firsts`와 `others`로 각 집단의 임신 기간을 나타내는 `Pmf`를 만듭니다.

In [41]:
first_pmf = Pmf.from_seq(firsts["prglngth"], name="firsts")
other_pmf = Pmf.from_seq(others["prglngth"], name="others")

첫아이와 다른 아이의 PMF를 막대그래프로 그리면 다음과 같습니다.

In [42]:
two_bar_plots(first_pmf, other_pmf)
decorate(xlabel="Weeks", ylabel="Probability", xlim=[20, 50])

도수분포표 대신 PMF를 그리면 표본 크기의 차이에 휘둘리지 않고 두 분포를 비교할 수 있습니다. 이 그림에서는 첫아이가 다른 아이들보다 39주에 태어날 가능성은 작고, 41주나 42주에 늦게 태어날 가능성은 커 보입니다.

## 다른 시각화 방법

도수분포표와 PMF는 데이터를 탐색하며 패턴과 관계를 찾을 때 유용합니다. 무슨 일이 일어나는지 어느 정도 파악했다면, 발견한 패턴이 최대한 분명히 드러나도록 시각화를 설계하는 것이 좋습니다.

NSFG 자료에서 두 분포의 가장 큰 차이는 최빈값 근처에 있습니다. 따라서 그 부분을 확대해 보는 것이 좋습니다. 다음 코드에서는 35주부터 45주까지 선택합니다. `range(35, 46)`은 끝값인 46을 포함하지 않습니다.

`Pmf`를 함수처럼 호출할 때는 값의 시퀀스를 넘겨 확률의 시퀀스를 얻을 수 있습니다.

In [43]:
weeks = range(35, 46)
first_pmf(weeks)

In [44]:
other_pmf(weeks)

따라서 다음과 같이 확률 차이를 구할 수 있습니다.

In [45]:
diffs = first_pmf(weeks) - other_pmf(weeks)
diffs

차이에 100을 곱해 퍼센트포인트 단위로 나타내면 다음과 같습니다.

In [46]:
plt.bar(weeks, diffs * 100)
decorate(xlabel="Weeks", ylabel="Difference (percentage points)")

이 그림에서는 패턴이 더 분명합니다. 첫아이는 39주에 태어날 가능성이 작고, 41주와 42주에 태어날 가능성은 조금 더 큽니다.

표본에서 이런 패턴을 보더라도 모집단에서도 성립하는지는 확신할 수 없습니다. 같은 모집단에서 다른 표본을 뽑아도 같은 패턴이 나타날지 모릅니다. 이 문제는 [9장](https://allendowney.github.io/ThinkStats/chap09.html)에서 다시 살펴보겠습니다.

## 용어 정리

이 장에는 앞 장들만큼 새로운 용어가 많지 않습니다.

- **정규화**(normalized): 확률의 합을 1로 맞추는 것.
- **확률질량함수**(probability mass function, PMF): 각 값을 그 확률에 대응시켜 분포를 나타내는 함수.

## 연습문제

이 장의 연습문제에서는 응답자 한 명당 한 행이 있는 NSFG 응답자 파일을 사용합니다. 자료를 내려받는 방법은 이 장의 노트북에 있습니다.

In [47]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemResp.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemResp.dat.gz")

이 데이터셋의 [코드북](https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/NSFG/Cycle6Codebook-Female.pdf)을 참고하세요.

`nsfg.py` 모듈에는 응답자 파일을 읽고 `DataFrame`을 반환하는 함수가 있습니다.

In [48]:
from nsfg import read_fem_resp

resp = read_fem_resp()
resp.shape

이 `DataFrame`에는 행이 7,643개, 열이 3,092개 있습니다.

### 연습문제 3.1

각 응답자가 “생존 출산한 아기의 수”를 기록한 `numbabes` 열을 선택하세요. `FreqTab` 객체를 만들어 이 열의 값별 도수를 표시하세요. 코드북의 도수와 일치하는지 확인하세요. `NaN`으로 바꿔야 할 특수한 값이 있나요?

이어서 `Pmf`를 만들고 막대그래프로 그리세요. 분포가 대칭인가요, 왼쪽으로 치우쳤나요, 오른쪽으로 치우쳤나요?

In [49]:
# Solution goes here

In [50]:
# Solution goes here

In [51]:
pmf.bar()
decorate(xlabel="Number of babies", ylabel="PMF")

In [52]:
# Solution goes here

### 연습문제 3.2

평균은 분포의 중심을 나타내고 분산은 퍼진 정도를 수치로 나타냅니다. 이와 비슷하게 **왜도**(skewness)는 분포가 왼쪽과 오른쪽 중 어느 쪽으로 치우쳤는지 나타내는 통계량입니다.

표본의 왜도는 편차를 세제곱한 값의 평균을 표준편차의 세제곱으로 나누어 구할 수 있습니다. 예를 들어 `numbabes`의 왜도는 다음과 같이 계산합니다.

In [53]:
numbabes = resp["numbabes"].replace(97, np.nan)

In [54]:
deviations = numbabes - numbabes.mean()
skewness = np.mean(deviations**3) / numbabes.std(ddof=0) ** 3
skewness

양수이면 오른쪽으로, 음수이면 왼쪽으로 치우친 분포입니다.

값의 시퀀스 대신 `Pmf`가 주어졌다면 다음 순서로 왜도를 계산할 수 있습니다.

1. `Pmf`의 각 값에서 평균을 빼 편차를 구합니다.
2. 편차를 세제곱하고 `Pmf`의 확률을 곱한 뒤 모두 더합니다.
3. 그 합을 표준편차의 세제곱으로 나눕니다.

`Pmf` 객체를 받아 왜도를 반환하는 `pmf_skewness` 함수를 작성하세요.

In [55]:
# Solution goes here

작성한 함수와 `numbabes`의 `Pmf`로 왜도를 구하세요. 앞에서 계산한 결과와 같은지 확인하세요.

In [56]:
# Solution goes here

### 연습문제 3.3

아이들에게 가족에 자녀가 몇 명인지 물어보는 조사에서도 수업 규모의 역설과 비슷한 일이 생깁니다. 자녀가 많은 가족은 표본에 들어갈 가능성이 더 높고, 자녀가 없는 가족은 표본에 들어갈 가능성이 전혀 없습니다.

`resp`에서 각 응답자의 가구에 사는 18세 미만 자녀 수를 기록한 `numkdhh`를 선택하세요. 이 열의 값으로 `Pmf`를 만드세요.

아이들을 조사해 본인을 포함한 18세 미만 자녀가 가구에 몇 명 있는지 물었다면 어떤 분포를 보게 될지, `bias` 함수로 계산하세요.

실제 분포와 편향된 분포를 그리고, 각각의 평균을 구하세요.

In [57]:
# Solution goes here

In [ ]:
# Solution goes here

In [59]:
# Solution goes here

In [60]:
# Solution goes here

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT](https://mit-license.org/)

본문·그림과 한국어 번역 라이선스: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)